In [0]:
from pyspark.sql.functions import expr, col, coalesce, lit, monotonically_increasing_id, when
from pyspark.sql import functions as F

In [0]:
# Load the raw table and check its size
df_init = spark.read.table("group3_gp.bronze.high_volume_fhv")

In [0]:
%sql
INSERT INTO group3_gp.gold.tabel_count
SELECT 
    'HVFHV' AS source,
    COUNT(*) AS row_count,
    'Ingest' AS reason
FROM group3_gp.bronze.high_volume_fhv;

In [0]:
# Remove unuseful columns that will not be used
cols_to_drop = {
    "sr_flag",
    "originating_base_num",
    "dispatching_base_num",
    "congestion_surcharge",
    "wav_request_flag",
    "wav_match_flag",
    "cbd_congestion_fee",
    "shared_request_flag",
    "access_a_ride_flag"   
}

df_clean_temp = df_init.drop(*cols_to_drop)
df_clean = df_clean_temp \
    .withColumnRenamed("tips", "tip_amount") \
    .withColumnRenamed("tolls", "tolls_amount") \
    .withColumnRenamed("trip_miles", "trip_distance") \
    .withColumnRenamed("sales_tax", "mta_tax")

In [0]:
# Convert every column from raw strings to proper types using try_cast

column_types = {
    "dolocationid": "integer",
    "dropoff_datetime": "timestamp",
    "hvfhs_license_num": "string",
    "pickup_datetime": "timestamp",
    "pulocationid": "integer",
    "Year": "integer",
    "base_passenger_fare": "decimal(10,2)",
    "bcf": "decimal(10,2)",
    "driver_pay": "decimal(10,2)",
    "on_scene_datetime": "timestamp",
    "originating_base_num": "string",
    "request_datetime": "timestamp",
    "sales_tax": "decimal(10,2)",
    "shared_match_flag": "string",
    "tip_amount": "decimal(10,2)",
    "tolls_amount": "decimal(10,2)",
    "trip_distance": "double",
    "trip_time": "double",
}

# Cast all columns to their proper types
for col, dtype in column_types.items():
    if col in df_clean.columns:
        df_clean = df_clean.withColumn(col, F.expr(f"try_cast({col} as {dtype})"))

# Tag every row as "High Volume FHV"

df = df_clean.withColumn("taxi_type", lit("Fhv"))

In [0]:
df = df.withColumn(
    "service_type",
    F.when(F.col("hvfhs_license_num") == "HV0002", "Juno")
     .when(F.col("hvfhs_license_num") == "HV0003", "Uber")
     .when(F.col("hvfhs_license_num") == "HV0004", "Via")
     .when(F.col("hvfhs_license_num") == "HV0005", "Lyft")
     .otherwise("Unknown")
)


In [0]:
# Drop rows that are clearly invalid:
# missing timestamps, dropoff before pickup, zero/negative fares,
final_df = df.filter(
    F.col("pickup_datetime").isNotNull() &
    F.col("dropoff_datetime").isNotNull() &
    (F.col("dropoff_datetime") > F.col("pickup_datetime")) &
    F.col("pulocationid").isNotNull() |
    F.col("dolocationid").isNotNull()
).drop(F.col("hvfhs_license_num"))

In [0]:
final_df.createOrReplaceTempView("final_df")

In [0]:
%sql
INSERT INTO group3_gp.gold.tabel_count
SELECT 
    'HVFHV_bronze' AS source,
    COUNT(*) AS row_count,
    'Drop rows missing crucial data, such as datetime, geodata, and impossible values' AS reason
FROM final_df;

In [0]:
# Create silver table 
final_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("group3_gp.silver.high_volume_fhv")